# Comparação Geral dos Datasets de Face Anti-Spoofing

## Objetivo

Este notebook consolida a análise dos quatro datasets atualmente disponíveis:

- **NUAA**
- **LCC-FASD**
- **CASIA-FASD**
- **AxonData Face Anti-Spoofing**

O objetivo não é juntar imediatamente todos os datasets num único conjunto de treino, mas comparar
as suas características, limitações e utilidade para a dissertação.

A análise considera:

- dimensão do dataset;
- distribuição `bona_fide` / `spoof`;
- splits disponíveis;
- sujeitos, quando identificáveis de forma fiável;
- diversidade de resoluções;
- tipos de ataques;
- diversidade de condições;
- limitações;
- papel recomendado de cada dataset na dissertação.

Este notebook evita `pandas`, `numpy` e `matplotlib`, para ser compatível com o ambiente atual.

## 1. Imports e localização automática dos relatórios CSV

In [15]:
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean, median
import csv
import math

TESE_ROOT = Path(
    r"C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE"
)

REPORT_FILENAMES = {
    "NUAA": "nuaa_dataset_report.csv",
    "LCC-FASD": "lcc_fasd_dataset_report.csv",
    "CASIA-FASD": "casia_fasd_dataset_report.csv",
    "AxonData": "axondata_fas_dataset_report.csv",
}

def find_report(filename):
    # Evita procurar dentro da .venv.
    candidates = [
        path for path in TESE_ROOT.rglob(filename)
        if ".venv" not in path.parts
    ]

    if not candidates:
        return None

    return candidates[0]

report_paths = {
    name: find_report(filename)
    for name, filename in REPORT_FILENAMES.items()
}

print("RELATÓRIOS ENCONTRADOS\n")

for name, path in report_paths.items():
    print(f"{name}: {path}")

missing = [
    name for name, path in report_paths.items()
    if path is None
]

if missing:
    raise FileNotFoundError(
        "Não foram encontrados os relatórios: "
        + ", ".join(missing)
    )

RELATÓRIOS ENCONTRADOS

NUAA: C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\nuaa_dataset_report.csv
LCC-FASD: C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\lcc_fasd_dataset_report.csv
CASIA-FASD: C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\casia_fasd_dataset_report.csv
AxonData: C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\axondata_fas_dataset_report.csv


## 2. Carregamento dos CSVs

In [16]:
def load_csv(path):
    with path.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        return list(csv.DictReader(file))

datasets = {
    name: load_csv(path)
    for name, path in report_paths.items()
}

for name, rows in datasets.items():
    print(f"{name}: {len(rows):,} registos")

NUAA: 12,614 registos
LCC-FASD: 18,827 registos
CASIA-FASD: 4,063 registos
AxonData: 149 registos


## 3. Funções auxiliares

In [17]:
def as_float(value):
    try:
        if value is None or value == "":
            return None
        return float(value)
    except (TypeError, ValueError):
        return None

def as_int(value):
    value = as_float(value)
    if value is None:
        return None
    return int(round(value))

def count_by(rows, field):
    return Counter(
        row.get(field, "")
        for row in rows
        if row.get(field, "") != ""
    )

def unique_nonempty(rows, field):
    return {
        row.get(field)
        for row in rows
        if row.get(field, "") != ""
    }

def resolutions(rows):
    values = set()

    for row in rows:
        width = as_int(row.get("width"))
        height = as_int(row.get("height"))

        if width is not None and height is not None:
            values.add((width, height))

    return values

def format_pct(n, total):
    return f"{(n / total * 100):.2f}%" if total else "N/A"

def markdown_table(headers, rows):
    line1 = "| " + " | ".join(str(x) for x in headers) + " |"
    line2 = "| " + " | ".join("---" for _ in headers) + " |"

    body = [
        "| " + " | ".join(str(x) for x in row) + " |"
        for row in rows
    ]

    return "\n".join([line1, line2] + body)

## 4. Comparação global

In [18]:
global_rows = []

for name, rows in datasets.items():
    total = len(rows)
    labels = count_by(rows, "label")

    bona = labels.get("bona_fide", 0)
    spoof = labels.get("spoof", 0)

    split_values = unique_nonempty(rows, "split")
    resolution_values = resolutions(rows)

    global_rows.append([
        name,
        total,
        bona,
        format_pct(bona, total),
        spoof,
        format_pct(spoof, total),
        len(resolution_values),
        ", ".join(sorted(split_values)) if split_values else "Sem split explícito",
    ])

print(
    markdown_table(
        [
            "Dataset",
            "Amostras",
            "Bona fide",
            "% bona fide",
            "Spoof",
            "% spoof",
            "Resoluções",
            "Splits"
        ],
        global_rows
    )
)

| Dataset | Amostras | Bona fide | % bona fide | Spoof | % spoof | Resoluções | Splits |
| --- | --- | --- | --- | --- | --- | --- | --- |
| NUAA | 12614 | 5105 | 40.47% | 7509 | 59.53% | 7 | test, train |
| LCC-FASD | 18827 | 1942 | 10.31% | 16885 | 89.69% | 1132 | LCC_FASD_development, LCC_FASD_evaluation, LCC_FASD_training |
| CASIA-FASD | 4063 | 995 | 24.49% | 3068 | 75.51% | 1 | test, train |
| AxonData | 149 | 24 | 16.11% | 125 | 83.89% | 9 | Sem split explícito |


### Leitura da comparação global

Os quatro relatórios representam unidades diferentes:

- NUAA e LCC-FASD são essencialmente conjuntos de imagens;
- a versão analisada do CASIA-FASD contém **frames RGB extraídos de vídeos**;
- AxonData contém uma mistura de **imagens e vídeos**.

Assim, o número de registos não deve ser interpretado diretamente como uma comparação do número
de vídeos ou sujeitos entre datasets.

## 5. Equilíbrio das classes

In [19]:
print("DISTRIBUIÇÃO DAS CLASSES\n")

for name, rows in datasets.items():
    labels = count_by(rows, "label")
    total = len(rows)

    print(name)

    for label in ["bona_fide", "spoof"]:
        count = labels.get(label, 0)
        print(
            f"  {label}: {count:,} "
            f"({format_pct(count, total)})"
        )

    print()

DISTRIBUIÇÃO DAS CLASSES

NUAA
  bona_fide: 5,105 (40.47%)
  spoof: 7,509 (59.53%)

LCC-FASD
  bona_fide: 1,942 (10.31%)
  spoof: 16,885 (89.69%)

CASIA-FASD
  bona_fide: 995 (24.49%)
  spoof: 3,068 (75.51%)

AxonData
  bona_fide: 24 (16.11%)
  spoof: 125 (83.89%)



### Interpretação

- **NUAA** é o dataset mais equilibrado dos quatro.
- **LCC-FASD** apresenta o desequilíbrio mais forte, com uma grande maioria de `spoof`.
- **CASIA-FASD** também favorece `spoof`, mas de forma menos extrema.
- **AxonData** é igualmente desequilibrado e, além disso, tem poucas amostras `bona_fide`.

Por esta razão, **accuracy não deve ser utilizada isoladamente** na avaliação dos modelos.

## 6. Splits disponíveis

In [20]:
for name, rows in datasets.items():
    if "split" not in rows[0]:
        print(f"{name}: sem coluna split\n")
        continue

    splits = defaultdict(Counter)

    for row in rows:
        split = row.get("split", "")
        label = row.get("label", "")

        if split:
            splits[split][label] += 1

    print(name)

    if not splits:
        print("  Sem split explícito")

    for split, counts in sorted(splits.items()):
        total = sum(counts.values())
        print(
            f"  {split}: {total:,} | "
            f"bona_fide={counts.get('bona_fide', 0):,} | "
            f"spoof={counts.get('spoof', 0):,}"
        )

    print()

NUAA
  test: 9,123 | bona_fide=3,362 | spoof=5,761
  train: 3,491 | bona_fide=1,743 | spoof=1,748

LCC-FASD
  LCC_FASD_development: 2,948 | bona_fide=405 | spoof=2,543
  LCC_FASD_evaluation: 7,580 | bona_fide=314 | spoof=7,266
  LCC_FASD_training: 8,299 | bona_fide=1,223 | spoof=7,076

CASIA-FASD
  test: 2,408 | bona_fide=591 | spoof=1,817
  train: 1,655 | bona_fide=404 | spoof=1,251

AxonData: sem coluna split



## 7. Resoluções e diversidade visual

A quantidade de resoluções diferentes ajuda a perceber a heterogeneidade da versão analisada.
Não mede, por si só, toda a diversidade de sensores, iluminação ou ambientes.

In [21]:
for name, rows in datasets.items():
    res = resolutions(rows)

    widths = [w for w, h in res]
    heights = [h for w, h in res]

    print(name)
    print("  Resoluções distintas:", len(res))

    if res:
        print("  Largura mínima:", min(widths))
        print("  Largura máxima:", max(widths))
        print("  Altura mínima:", min(heights))
        print("  Altura máxima:", max(heights))

    counts = Counter()

    for row in rows:
        width = as_int(row.get("width"))
        height = as_int(row.get("height"))

        if width is not None and height is not None:
            counts[(width, height)] += 1

    print("  Resoluções mais frequentes:")

    for (width, height), count in counts.most_common(5):
        print(f"    {width}x{height}: {count:,}")

    print()

NUAA
  Resoluções distintas: 7
  Largura mínima: 103
  Largura máxima: 393
  Altura mínima: 103
  Altura máxima: 393
  Resoluções mais frequentes:
    162x162: 4,830
    203x203: 3,871
    252x252: 1,791
    131x131: 1,294
    317x317: 414

LCC-FASD
  Resoluções distintas: 1132
  Largura mínima: 59
  Largura máxima: 2112
  Altura mínima: 59
  Altura máxima: 2112
  Resoluções mais frequentes:
    400x400: 132
    427x427: 126
    401x401: 109
    453x453: 97
    373x373: 97

CASIA-FASD
  Resoluções distintas: 1
  Largura mínima: 256
  Largura máxima: 256
  Altura mínima: 256
  Altura máxima: 256
  Resoluções mais frequentes:
    256x256: 4,063

AxonData
  Resoluções distintas: 9
  Largura mínima: 811
  Largura máxima: 3264
  Altura mínima: 1080
  Altura máxima: 3840
  Resoluções mais frequentes:
    2160x3840: 74
    1080x1920: 47
    1000x1333: 21
    1000x1336: 2
    1000x2222: 1



### Interpretação

- **LCC-FASD** tem, de longe, a maior diversidade de resoluções na versão analisada.
- **NUAA** contém várias resoluções, mas todas são faces já recortadas e relativamente pequenas.
- **CASIA-FASD** está completamente normalizado para `256x256`, portanto a resolução não representa
  a diversidade original dos vídeos.
- **AxonData** contém material de resolução elevada, sobretudo vídeo, mas é uma amostra pequena.

## 8. NUAA — análise específica

In [22]:
nuaa = datasets["NUAA"]

subjects = unique_nonempty(nuaa, "subject")

train_subjects = {
    row["subject"]
    for row in nuaa
    if row.get("split") == "train"
}

test_subjects = {
    row["subject"]
    for row in nuaa
    if row.get("split") == "test"
}

overlap = train_subjects & test_subjects

sessions = defaultdict(Counter)

for row in nuaa:
    session = row.get("session", "")
    split = row.get("split", "")

    if session and split:
        sessions[session][split] += 1

print("Sujeitos identificados:", len(subjects))
print("Sujeitos no train:", len(train_subjects))
print("Sujeitos no test:", len(test_subjects))
print("Sujeitos presentes em ambos:", len(overlap))

print("\nSESSÕES\n")

for session, counts in sorted(sessions.items()):
    print(f"Session {session}: {dict(counts)}")

Sujeitos identificados: 16
Sujeitos no train: 9
Sujeitos no test: 16
Sujeitos presentes em ambos: 9

SESSÕES

Session 01: {'train': 1744}
Session 02: {'train': 1747}
Session 03: {'test': 9123}


### NUAA — conclusão

A versão analisada é simples e adequada para validar a pipeline inicial de classificação
`bona_fide` vs. `spoof`.

A divisão treino/teste introduz uma mudança de sessão, mas **não é subject-disjoint**: existem
identificadores de sujeitos presentes nos dois conjuntos. Por isso, não deve ser usada como
principal evidência de generalização para identidades desconhecidas.

**Papel recomendado:** dataset auxiliar / baseline inicial.

## 9. LCC-FASD — análise específica

In [23]:
lcc = datasets["LCC-FASD"]

split_counts = defaultdict(Counter)

for row in lcc:
    split = row.get("split", "")
    label = row.get("label", "")

    if split:
        split_counts[split][label] += 1

for split, counts in sorted(split_counts.items()):
    total = sum(counts.values())
    spoof = counts.get("spoof", 0)

    print(split)
    print("  Total:", f"{total:,}")
    print("  Bona fide:", f"{counts.get('bona_fide', 0):,}")
    print("  Spoof:", f"{spoof:,}")
    print("  Spoof (%):", format_pct(spoof, total))
    print()

print(
    "Resoluções distintas:",
    len(resolutions(lcc))
)

LCC_FASD_development
  Total: 2,948
  Bona fide: 405
  Spoof: 2,543
  Spoof (%): 86.26%

LCC_FASD_evaluation
  Total: 7,580
  Bona fide: 314
  Spoof: 7,266
  Spoof (%): 95.86%

LCC_FASD_training
  Total: 8,299
  Bona fide: 1,223
  Spoof: 7,076
  Spoof (%): 85.26%

Resoluções distintas: 1132


### LCC-FASD — conclusão

O LCC-FASD acrescenta uma grande diversidade de resoluções e condições de imagem, mas apresenta
um **forte desequilíbrio de classes**.

A metadata exportada não permite determinar de forma uniforme sujeitos, dispositivos e tipos de
ataque para todas as amostras. Assim, não devemos inventar essa informação a partir dos nomes dos
ficheiros.

**Papel recomendado:** dataset auxiliar para robustez a qualidade/resolução e para testar o efeito
de forte class imbalance.

## 10. CASIA-FASD — análise específica

In [24]:
casia = datasets["CASIA-FASD"]

subjects_by_split = defaultdict(set)

for row in casia:
    split = row.get("split", "")
    subject = row.get("subject", "")

    if split and subject:
        subjects_by_split[split].add(subject)

print("SUJEITOS\n")

for split, subjects in sorted(subjects_by_split.items()):
    print(f"{split}: {len(subjects)}")

# Os IDs são locais a cada split.
estimated_distinct_subjects = sum(
    len(subjects)
    for subjects in subjects_by_split.values()
)

print(
    "Total de identidades considerando IDs locais aos splits:",
    estimated_distinct_subjects
)

print("\nTIPOS DE ATAQUE\n")

attacks = Counter(
    row.get("attack_type", "") or "none"
    for row in casia
)

for attack, count in attacks.most_common():
    print(f"{attack}: {count:,}")

print("\nQUALIDADE\n")

qualities = count_by(casia, "quality")

for quality, count in qualities.most_common():
    print(f"{quality}: {count:,}")

print("\nATAQUE x QUALIDADE\n")

attack_quality = defaultdict(Counter)

for row in casia:
    attack = row.get("attack_type", "") or "none"
    quality = row.get("quality", "") or "unknown"
    attack_quality[attack][quality] += 1

for attack, counts in sorted(attack_quality.items()):
    print(attack, dict(counts))

SUJEITOS

test: 30
train: 20
Total de identidades considerando IDs locais aos splits: 50

TIPOS DE ATAQUE

warped_photo: 1,185
video_replay: 1,004
none: 995
cut_photo: 879

QUALIDADE

high: 1,480
normal: 1,294
low: 1,289

ATAQUE x QUALIDADE

cut_photo {'normal': 283, 'low': 350, 'high': 246}
none {'normal': 291, 'low': 314, 'high': 390}
video_replay {'normal': 360, 'low': 271, 'high': 373}
warped_photo {'normal': 360, 'low': 354, 'high': 471}


### CASIA-FASD — conclusão

Dos datasets atualmente disponíveis, o CASIA-FASD é o mais importante para o protocolo científico
principal.

A versão analisada permite relacionar cada frame com:

- sujeito;
- split;
- tipo de ataque;
- nível de qualidade;
- vídeo de origem;
- número do frame.

Os identificadores numéricos de sujeito são **locais a cada split**. Por isso, um `subject=1` no
train não representa o mesmo indivíduo que um `subject=1` no test. Considerando esta regra, existem
20 identidades de treino e 30 de teste, num total de 50 identidades distintas.

**Papel recomendado:** dataset principal já disponível e componente `C` do futuro benchmark MCIO.

## 11. AxonData — análise específica

In [25]:
axon = datasets["AxonData"]

media = count_by(axon, "media_type")
attacks = count_by(axon, "attack_type")
categories = count_by(axon, "category")

print("MEDIA\n")
for media_type, count in media.most_common():
    print(f"{media_type}: {count}")

print("\nTIPOS DE ATAQUE / CATEGORIAS NORMALIZADAS\n")
for attack, count in attacks.most_common():
    print(f"{attack}: {count}")

fps_values = [
    as_float(row.get("fps"))
    for row in axon
    if as_float(row.get("fps")) is not None
]

duration_values = [
    as_float(row.get("duration_seconds"))
    for row in axon
    if as_float(row.get("duration_seconds")) is not None
]

frame_values = [
    as_float(row.get("frame_count"))
    for row in axon
    if as_float(row.get("frame_count")) is not None
]

if fps_values:
    print("\nVÍDEOS")
    print(f"  FPS mínimo: {min(fps_values):.2f}")
    print(f"  FPS máximo: {max(fps_values):.2f}")
    print(f"  FPS médio: {mean(fps_values):.2f}")

if duration_values:
    print(f"  Duração mínima: {min(duration_values):.2f} s")
    print(f"  Duração máxima: {max(duration_values):.2f} s")
    print(f"  Duração média: {mean(duration_values):.2f} s")

if frame_values:
    print(f"  Frames mínimos: {int(min(frame_values))}")
    print(f"  Frames máximos: {int(max(frame_values))}")
    print(f"  Frames médios: {mean(frame_values):.2f}")

MEDIA

video: 120
image: 29

TIPOS DE ATAQUE / CATEGORIAS NORMALIZADAS

3d_paper_mask: 36
none: 24
textile_3d_mask: 23
cutout: 15
silicone_mask: 11
latex_mask: 10
replay_display: 10
replay_mobile: 10
wrapped_3d_paper_mask: 10

VÍDEOS
  FPS mínimo: 29.99
  FPS máximo: 60.05
  FPS médio: 31.29
  Duração mínima: 3.40 s
  Duração máxima: 18.73 s
  Duração média: 9.31 s
  Frames mínimos: 102
  Frames máximos: 909
  Frames médios: 296.18


### Verificação importante do AxonData

In [26]:
possible_real_auxiliary = []

for row in axon:
    relative_path = (
        row.get("relative_path", "")
        .replace("\\", "/")
    )

    if "/Real/" in f"/{relative_path}":
        possible_real_auxiliary.append(row)

print(
    "Ficheiros encontrados em subpastas denominadas 'Real':",
    len(possible_real_auxiliary)
)

for row in possible_real_auxiliary[:20]:
    print(
        row.get("relative_path"),
        "| label atual:",
        row.get("label")
    )

Ficheiros encontrados em subpastas denominadas 'Real': 5
Replay_display_attacks\Real\1.jpg | label atual: spoof
Replay_display_attacks\Real\2.jpg | label atual: spoof
Replay_display_attacks\Real\3.jpg | label atual: spoof
Replay_display_attacks\Real\4.jpg | label atual: spoof
Replay_display_attacks\Real\5.jpg | label atual: spoof


### AxonData — conclusão

O AxonData acrescenta aquilo que falta mais claramente aos restantes datasets: **grande variedade de
PAIs**, sobretudo ataques com máscaras 3D, latex, silicone, tecido, papel e ataques de replay.

Contudo, a versão pública é pequena e não possui um protocolo académico de treino/teste comparável ao
CASIA ou ao futuro MCIO. Além disso, existe uma diferença importante entre modalidades: as `Selfies`
são imagens, enquanto a maioria dos ataques é vídeo.

Existe ainda uma verificação necessária antes de qualquer treino: a estrutura
`Replay_display_attacks/Real` contém ficheiros que, no relatório atual, herdaram a label da categoria.
Antes de estes ficheiros entrarem numa experiência, deve ser confirmado se são apresentações
`bona_fide`, imagens fonte utilizadas para gerar o replay, ou apenas material auxiliar.

**Papel recomendado:** dataset auxiliar para testes de ataques não vistos e robustez a novos PAIs,
não benchmark principal.

## 12. Comparação conceptual

In [27]:
comparison_rows = [
    [
        "NUAA",
        "Média",
        "Relativamente bom",
        "Baixa",
        "Baixa/média",
        "Sessões, mas sujeitos sobrepostos",
        "Baseline / auxiliar"
    ],
    [
        "LCC-FASD",
        "Alta",
        "Muito desequilibrado",
        "Muito alta",
        "Metadata limitada",
        "Train / dev / evaluation",
        "Robustez / auxiliar"
    ],
    [
        "CASIA-FASD",
        "Média",
        "Desequilibrado",
        "Controlada (256x256)",
        "Boa: photo, cut, replay + quality",
        "Train/test com identidades distintas",
        "Principal / MCIO"
    ],
    [
        "AxonData",
        "Muito baixa",
        "Muito desequilibrado",
        "Alta no material original",
        "Muito alta, sobretudo máscaras",
        "Sem split académico",
        "Ataques não vistos / auxiliar"
    ],
]

print(
    markdown_table(
        [
            "Dataset",
            "Dimensão",
            "Balanceamento",
            "Diversidade visual",
            "Diversidade de ataques",
            "Protocolo",
            "Papel"
        ],
        comparison_rows
    )
)

| Dataset | Dimensão | Balanceamento | Diversidade visual | Diversidade de ataques | Protocolo | Papel |
| --- | --- | --- | --- | --- | --- | --- |
| NUAA | Média | Relativamente bom | Baixa | Baixa/média | Sessões, mas sujeitos sobrepostos | Baseline / auxiliar |
| LCC-FASD | Alta | Muito desequilibrado | Muito alta | Metadata limitada | Train / dev / evaluation | Robustez / auxiliar |
| CASIA-FASD | Média | Desequilibrado | Controlada (256x256) | Boa: photo, cut, replay + quality | Train/test com identidades distintas | Principal / MCIO |
| AxonData | Muito baixa | Muito desequilibrado | Alta no material original | Muito alta, sobretudo máscaras | Sem split académico | Ataques não vistos / auxiliar |


## 13. Papel de cada dataset na dissertação

### NUAA

Utilizar para:

- validar rapidamente carregamento, pré-processamento e classificação;
- estabelecer um baseline simples;
- estudar uma mudança de sessão.

Não utilizar como principal prova de generalização subject-independent.

### LCC-FASD

Utilizar para:

- testar robustez a grande variação de resolução/qualidade;
- estudar impacto de class imbalance;
- avaliar se o modelo mantém comportamento consistente num dataset visualmente heterogéneo.

### CASIA-FASD

Utilizar para:

- experiências principais já disponíveis;
- comparação entre `warped_photo`, `cut_photo` e `video_replay`;
- análise por `low`, `normal` e `high` quality;
- integração futura no benchmark MCIO.

### AxonData

Utilizar para:

- avaliar ataques diferentes dos utilizados no treino;
- testar generalização para máscaras e PAIs 3D;
- investigação auxiliar de robustez.

Não utilizar diretamente como benchmark principal sem primeiro definir um protocolo controlado.

## 14. Estratégia experimental recomendada

Nesta fase, **não se recomenda juntar simplesmente todas as amostras dos quatro datasets num único
treino**. Isso destruiria parte da informação de domínio e poderia introduzir confusões devido às
diferenças entre:

- imagens e vídeos;
- frames extraídos e vídeos originais;
- resoluções;
- balanceamento;
- protocolos;
- tipos de ataque.

A estratégia recomendada é:

1. preservar o protocolo nativo de cada dataset;
2. normalizar apenas a representação necessária para o modelo;
3. criar um baseline intra-dataset;
4. avaliar por tipo de ataque e condição quando a metadata permite;
5. avançar para cross-dataset quando estiverem disponíveis Replay-Attack, MSU-MFSD e OULU-NPU;
6. utilizar AxonData como teste adicional de generalização para PAIs não vistos.

## 15. Métricas recomendadas

Devido ao desequilíbrio das classes, `accuracy` não deve ser a métrica principal.

Para PAD devem ser priorizadas:

- **APCER** — proporção de ataques classificados como bona fide;
- **BPCER** — proporção de bona fide classificados como ataque;
- **ACER** — média de APCER e BPCER;
- **HTER** — especialmente útil em experiências cross-domain;
- **AUC** — análise independente de um único threshold.

`Balanced Accuracy` pode ser apresentada como métrica auxiliar, mas não substitui as métricas
específicas de PAD.

## 16. Exportação de uma tabela-resumo

In [28]:
OUTPUT_DIR = TESE_ROOT / "Notebooks" / "results_datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / "dataset_comparison_summary.csv"

summary_rows = []

for name, rows in datasets.items():
    total = len(rows)
    labels = count_by(rows, "label")
    res = resolutions(rows)

    summary_rows.append({
        "dataset": name,
        "samples": total,
        "bona_fide": labels.get("bona_fide", 0),
        "spoof": labels.get("spoof", 0),
        "bona_fide_pct": round(
            labels.get("bona_fide", 0) / total * 100,
            2
        ) if total else "",
        "spoof_pct": round(
            labels.get("spoof", 0) / total * 100,
            2
        ) if total else "",
        "unique_resolutions": len(res),
        "splits": "; ".join(
            sorted(unique_nonempty(rows, "split"))
        ),
    })

with summary_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(summary_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(summary_rows)

print("Resumo exportado para:")
print(summary_path)

Resumo exportado para:
C:\Users\luisc\Desktop\Faculdade\Cadeiras\TESE\Notebooks\results_datasets\dataset_comparison_summary.csv


## Conclusão geral

Os quatro datasets atualmente disponíveis são complementares, mas não equivalentes.

O **NUAA** é útil como baseline simples e para validação inicial da pipeline. O **LCC-FASD** acrescenta
forte diversidade de resolução e permite estudar o impacto do desequilíbrio de classes. O
**CASIA-FASD** é, neste momento, o dataset com maior importância experimental para a dissertação,
porque possui um protocolo mais estruturado, identidades distintas entre treino e teste e metadata
explícita sobre diferentes tipos de ataque e níveis de qualidade. O **AxonData** acrescenta uma
diversidade de PAIs muito superior, sobretudo máscaras e ataques 3D, sendo adequado como dataset
auxiliar para testar ataques não vistos.

A principal conclusão é que nenhum dos quatro datasets deve ser usado isoladamente para demonstrar
generalização robusta. A dissertação deverá combinar experiências intra-dataset com avaliações
cross-domain, mantendo o **MCIO (MSU-MFSD, CASIA-FASD, Idiap Replay-Attack e OULU-NPU)** como objetivo
principal assim que os restantes acessos forem obtidos.

Até lá, os quatro datasets atuais já permitem validar a pipeline, construir um baseline, estudar
class imbalance, qualidade/resolução, diferentes PAIs e preparar o protocolo experimental final.